# Rekurencyjne sieci neuronowe (RNN): od równań do praktyki

> **Cel:** zbudować trwały punkt odniesienia do RNN w PyTorch: zrozumieć rekurencję i BPTT, odróżniać vanilla RNN, GRU i LSTM, prawidłowo przygotowywać sekwencje, trenować model językowy i diagnozować typowe problemy.
>
> **Poziom:** zakładam znajomość podstaw ML, gradientu i sieci feed-forward. Wzory są połączone z kształtami tensorów i kodem.
>
> **Praktyczny projekt:** znakowy model językowy przewidujący kolejny znak na publicznym korpusie Tiny Shakespeare. Kod ogranicza się do PyTorch i biblioteki standardowej Pythona.

Ten notebook jest samodzielnym przewodnikiem. Komórki można wykonywać po kolei; trening ma mały domyślny budżet, żeby dało się szybko sprawdzić cały przepływ. Zwiększ `MAX_STEPS`, gdy chcesz uzyskać lepsze próbki.


## Mapa pojęć

1. **Sekwencja i stan:** RNN przetwarza elementy po kolei i przenosi stan niosący informację z przeszłości.
2. **Uczenie:** parametry są współdzielone w czasie; gradient przepływa przez rozwiniętą w czasie sieć (BPTT).
3. **Problem długich zależności:** iloczyny Jacobianów mogą wygaszać albo eksplodować gradient. LSTM i GRU wprowadzają bramki.
4. **Praktyka:** kodowanie tokenów, przesunięte pary wejście/cel, batchowanie fragmentów, clipping, ewaluacja i generowanie.
5. **Wybór modelu:** RNN jest prosty i szybki dla krótkich zależności; GRU często daje dobry kompromis; LSTM ma jawny stan komórki i bramkę zapominania.

### Oczekiwane kształty

Dla batcha sekwencji o długości `T` i wymiarze cech `F`: wejście `x` ma zwykle kształt **`(B, T, F)`** (`batch_first=True`). Ukryty stan to **`(num_layers * directions, B, H)`**. Wyjście z warstwy rekurencyjnej ma kształt **`(B, T, directions * H)`**. W tym notebooku tokeny są indeksami całkowitymi `(B,T)`, a embedding zamienia je na wektory `(B,T,E)`.


## 1. Co sprawia, że sieć jest rekurencyjna?

W sieci feed-forward wynik zależy od bieżącego wejścia. RNN dodaje **stan ukryty** `h_t`, który streszcza dotychczasowy kontekst:

$$
a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h,\qquad h_t = \phi(a_t)
$$
$$
y_t = W_{hy}h_t+b_y.
$$

Te same macierze `W_xh`, `W_hh`, `W_hy` są używane w każdym kroku czasu. To współdzielenie parametrów sprawia, że model może obsługiwać sekwencje o różnych długościach. `h_0` zwykle inicjalizujemy zerami albo uczonym wektorem.

Ważna intuicja: rekurencja nie oznacza, że model „pamięta wszystko”. Stan ma skończony wymiar i jest aktualizowany stratnie. To, co zostaje w stanie, musi być wyuczone.


In [ ]:
import math
import random
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Urządzenie:", device)


### Rozwinięcie w czasie (unrolling)

Dla sekwencji długości `T` można narysować RNN jako `T` kopii tej samej komórki. To nie są osobne parametry: wszystkie kopie współdzielą wagi. Każdy krok może emitować predykcję, a funkcja straty może sumować błędy po czasie.

- **Many-to-one:** sekwencja → jedna etykieta (np. sentyment); zwykle klasyfikujemy końcowy stan.
- **One-to-many:** jeden kontekst → sekwencja (np. generowanie); stan/początkowy token warunkuje kolejne kroki.
- **Many-to-many:** wejściowa sekwencja → predykcja w każdym kroku (tagowanie, model językowy).

W modelu językowym uczymy `P(x_t | x_{<t})`. Jedna sekwencja wejściowa dostarcza kontekst, a wyjście w każdym kroku jest rozkładem nad następnym tokenem.


## 2. BPTT i długie zależności

**Backpropagation Through Time (BPTT)** to zwykłe różniczkowanie wstecz po rozwiniętym grafie obliczeń. Współdzielona macierz rekurencyjna dostaje wkład gradientu z wielu kroków.

W uproszczeniu gradient dalekiego kroku zawiera iloczyn Jacobianów:

$$
\frac{\partial h_t}{\partial h_k} = \prod_{j=k+1}^{t}\frac{\partial h_j}{\partial h_{j-1}}.
$$

Jeśli normy kolejnych czynników są zwykle mniejsze od 1, gradient zanika; jeśli większe od 1, może eksplodować. Wysoka/niska norma nie jest jedynym czynnikiem (liczy się też kierunek i nieliniowość), ale intuicja iloczynu jest kluczowa.

**Typowe odpowiedzi praktyczne:** bramkowane komórki (LSTM/GRU), sensowna długość sekwencji, normalizacja/initializacja, clipping normy gradientu. Clipping ogranicza skok aktualizacji, lecz nie naprawia reprezentacji ani sam z siebie nie tworzy długiej pamięci.

### Truncated BPTT

Przy długich strumieniach dzielimy sekwencję na fragmenty i odłączamy stan między fragmentami (`state = state.detach()`). Stan nadal przenosi kontekst do przodu, ale gradient nie cofa się przez granicę fragmentu. Zmniejsza to pamięć i koszt, kosztem ograniczenia horyzontu uczenia.


## 3. Rodzina komórek rekurencyjnych

### Vanilla RNN

Najprostsza aktualizacja $$h_t = tanh(W_x x_t + W_h h_{t-1} + b)$$. Dobra jako punkt odniesienia i dla krótkich zależności; trudna w uczeniu długiego kontekstu.

### LSTM

LSTM utrzymuje **stan ukryty** `h_t` oraz **stan komórki** `c_t`. Bramki sterują zapominaniem, dopisywaniem i odczytem:

$$
i_t=\sigma(W_i[x_t,h_{t-1}]+b_i),\quad f_t=\sigma(W_f[x_t,h_{t-1}]+b_f),\quad o_t=\sigma(W_o[x_t,h_{t-1}]+b_o)
$$
$$
\tilde c_t=\tanh(W_c[x_t,h_{t-1}]+b_c),\quad c_t=f_t\odot c_{t-1}+i_t\odot\tilde c_t,\quad h_t=o_t\odot\tanh(c_t).
$$

Ścieżka addytywna w `c_t` ułatwia przepływ gradientu. Bramki są uczone, nie są ręcznie ustawioną pamięcią semantyczną.

### GRU

GRU łączy pamięć w jednym stanie `h_t` i używa bramki aktualizacji oraz resetu. Ma mniej parametrów niż LSTM w typowych konfiguracjach i często działa podobnie dobrze. Nie ma uniwersalnego zwycięzcy — porównuj na własnym zadaniu, z podobnym budżetem strojenia.

### Dwukierunkowe i wielowarstwowe

- `num_layers > 1` buduje stos komórek; dropout w `nn.RNN/GRU/LSTM` działa pomiędzy warstwami (i jest ignorowany dla jednej warstwy).
- `bidirectional=True` przetwarza sekwencję w obu kierunkach. Dobre dla klasyfikacji/tagowania, gdy cała sekwencja jest dostępna. **Nie nadaje się do przyczynowego generowania**, bo kierunek wsteczny widzi przyszłość.


## 4. Przygotowanie środowiska i publicznego zbioru

Korpus jest pobierany z publicznego repozytorium [karpathy/char-rnn](https://github.com/karpathy/char-rnn). Plik zostanie zapisany w katalogu `data/` obok notebooka. Pierwsze wykonanie wymaga dostępu do sieci; przy braku dostępu można umieścić tam plik `tinyshakespeare.txt` ręcznie.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
print("Pobieram Tiny Shakespeare do pamięci…")
with urllib.request.urlopen(DATA_URL) as response:
    text = response.read().decode("utf-8")
print(f"Znaki: {len(text):,}")
print(text[:500])


## 5. Tokenizacja znakowa i podział danych

Tokenizacja znakowa jest prosta i uwidacznia autoregresję bez dodatkowej biblioteki. Słownik ma tylko znaki występujące w korpusie; w języku naturalnym produkcyjnym zwykle używa się subword-tokenizera.

Podział jest **sekwencyjny**: pierwsze 90% trening, pozostałe 10% walidacja. To odzwierciedla predykcję przyszłego tekstu i zapobiega losowemu mieszaniu sąsiednich fragmentów między zbiorami. Tu nie ma osobnego testu — celem jest studium modelu, nie benchmark publikacyjny.


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encoded = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

split = int(0.9 * len(encoded))
train_ids = encoded[:split]
val_ids = encoded[split:]
print("Vocab size:", vocab_size)
print("Train / val tokens:", len(train_ids), "/", len(val_ids))
print("Przykładowe ID:", train_ids[:20].tolist())


### Zadanie predykcyjne bez przecieku

Jeżeli fragment ma tokeny `a b c d`, wejściem są `a b c`, a celami `b c d`. Model przewiduje następny znak na każdej pozycji. Cross-entropy porównuje logits `(B,T,V)` z etykietami `(B,T)`; przed stratą spłaszczamy dwa pierwsze wymiary.

Poniższy generator losuje początki fragmentów wewnątrz jednego podziału. Dzięki przesunięciu o jeden nie odrywamy kontekstu w obrębie fragmentu.


In [ ]:
BATCH_SIZE = 64
BLOCK_SIZE = 128


def get_batch(split_name, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE):
    source = train_ids if split_name == "train" else val_ids
    starts = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", batch_size=2, block_size=12)
print("x:", xb.shape, "y:", yb.shape)
print("wejście:", "".join(itos[i] for i in xb[0].tolist()))
print("cel:    ", "".join(itos[i] for i in yb[0].tolist()))


## 6. Model w PyTorch

`nn.RNN`, `nn.GRU` i `nn.LSTM` mają wspólny interfejs w PyTorch. Wbudowana implementacja jest zoptymalizowana; warto jednak znać równania, aby rozumieć co oznacza `hidden_size`, stan i kształty.

Architektura poniżej: **embedding → komórka rekurencyjna → projekcja do słownika**. Logits to nieprawdopodobieństwa; `CrossEntropyLoss` wykonuje stabilny log-softmax wewnętrznie. Nie dodawaj `softmax` przed tą stratą.


In [ ]:
class CharRNN(nn.Module):
    def __init__(self, cell="LSTM", embedding_dim=64, hidden_size=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        cell_cls = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[cell]
        self.rnn = cell_cls(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, vocab_size)
        self.cell_name = cell

    def forward(self, idx, state=None):
        emb = self.embedding(idx)              # (B, T) -> (B, T, E)
        out, state = self.rnn(emb, state)      # (B, T, H)
        logits = self.head(out)                # (B, T, V)
        return logits, state

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregresywnie dopisuje tokeny; idx ma kształt (B, kontekst)."""
        self.eval()
        for _ in range(max_new_tokens):
            context = idx[:, -BLOCK_SIZE:]
            logits, _ = self(context)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                k = min(top_k, logits.size(-1))
                threshold = torch.topk(logits, k).values[:, [-1]]
                logits = logits.masked_fill(logits < threshold, float("-inf"))
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

model = CharRNN().to(device)
logits, state = model(xb)
print("logits:", logits.shape)
if isinstance(state, tuple):
    print("h_n, c_n:", state[0].shape, state[1].shape)
else:
    print("h_n:", state.shape)
print("Parametry:", f"{sum(p.numel() for p in model.parameters()):,}")


### Interfejs stanu i pułapki kształtów

- `nn.LSTM` zwraca `(output, (h_n, c_n))`; `nn.RNN/GRU` zwracają `(output, h_n)`.
- Przy `batch_first=True` zmienia się kształt `input` i `output`, **nie** ukrytego stanu.
- Dla dwukierunkowości wymiar kierunków jest pierwszy w stanie, a ostatni w `output` jest `directions * hidden_size`.
- W tym zadaniu każdy batch to niezależne okna, więc stan startuje domyślnie od zera. W stanie strumieniowym można przekazywać go między fragmentami, ale trzeba pilnować kolejności przykładów oraz odłączać graf przez `.detach()`.
- `pack_padded_sequence` / `pad_packed_sequence` przydają się przy batchu sekwencji o różnych długościach. Alternatywnie zamaskuj padding w funkcji straty (`ignore_index`). Tutaj wszystkie okna mają tę samą długość.


## 7. Trening, ewaluacja i diagnostyka

Perplexity modelu językowego to `exp(średnia cross-entropy)` i można ją intuicyjnie czytać jako efektywną liczbę możliwych tokenów. Porównuj ją tylko przy tym samym tokenizatorze i podziale danych.

Pętla używa AdamW i clippingu globalnej normy gradientu. Szacowanie straty odbywa się w `eval()` i `no_grad()`, na kilku losowych batchach. Mały budżet kroków pokazuje działający przepływ, ale nie jest pełnym strojeniem.


In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_batches=20):
    model.eval()
    result = {}
    for split_name in ("train", "val"):
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch(split_name)
            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
            losses.append(loss.item())
        result[split_name] = sum(losses) / len(losses)
    model.train()
    return result


def train_model(cell, max_steps=120, eval_interval=30, lr=3e-3):
    net = CharRNN(cell=cell).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr)
    history = {"step": [], "train": [], "val": []}
    start_time = time.time()
    net.train()
    for step in range(max_steps + 1):
        if step % eval_interval == 0 or step == max_steps:
            losses = estimate_loss(net)
            history["step"].append(step)
            history["train"].append(losses["train"])
            history["val"].append(losses["val"])
            print(f"{cell:4s} | krok {step:4d} | train {losses['train']:.3f} | val {losses['val']:.3f}")
        if step == max_steps:
            break
        x, y = get_batch("train")
        logits, _ = net(x)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
    print(f"czas: {time.time() - start_time:.1f}s")
    return net, history


### Porównanie RNN, GRU i LSTM

Używamy tej samej szerokości, embeddingu, podziału danych i budżetu kroków. Liczba parametrów nie będzie identyczna, bo komórki mają różną liczbę bramek. To szybkie porównanie edukacyjne, nie rozstrzygający eksperyment: dla rzetelnego wniosku powtórz kilka seedów, dobierz hiperparametry i uśrednij wyniki.

Na CPU trening trzech wariantów może potrwać chwilę. Możesz zmniejszyć `MAX_STEPS` do 40 na szybki smoke-run albo zwiększyć do kilkuset dla ciekawszych próbek.


In [ ]:
MAX_STEPS = 120
models = {}
histories = {}
for cell in ("RNN", "GRU", "LSTM"):
    models[cell], histories[cell] = train_model(cell, max_steps=MAX_STEPS)


In [ ]:
plt.figure(figsize=(9, 5))
for cell, hist in histories.items():
    plt.plot(hist["step"], hist["val"], marker="o", label=cell)
plt.xlabel("Krok optymalizacji")
plt.ylabel("Walidacyjna cross-entropy (nats/znak)")
plt.title("Porównanie komórek rekurencyjnych")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

for cell, net in models.items():
    print(f"{cell}: {sum(p.numel() for p in net.parameters()):,} parametrów")


### Jak czytać krzywe?

- Malejący trening i walidacja sugerują, że model wykorzystuje lokalne regularności tekstu.
- Rosnąca luka trening–walidacja może oznaczać przeuczenie; ogranicz pojemność, dodaj dropout, skróć trening lub zbierz więcej danych.
- Niewielka różnica między komórkami przy małym budżecie nie dowodzi równoważności. Optymalizacja i seedy wnoszą szum.
- Dla losowego predyktora o równym rozkładzie oczekiwana strata to `log(vocab_size)`. Model uczący się powinien zejść niżej.

Warto również śledzić normę gradientu, czas na krok, pamięć GPU i jakość próbek. Sama strata nie mówi wszystkiego o użyteczności tekstu.


## 8. Generowanie i temperatura

Generowanie zaczyna się od znaku startowego. W każdym kroku model zwraca logits kolejnego znaku; softmax daje rozkład, a `multinomial` losuje token. Temperatura `τ` przekształca logits `z/τ`: niska temperatura wzmacnia najbardziej prawdopodobne opcje, wysoka spłaszcza rozkład. `top_k` zostawia tylko `k` najbardziej prawdopodobnych tokenów.

Sampling zmienia różnorodność, ale nie wiedzę modelu. Greedy decoding (`argmax`) jest deterministyczny i często zapętla się.


In [ ]:
def sample_text(cell="LSTM", prompt="ROMEO:", length=800, temperature=0.8, top_k=20):
    net = models[cell]
    prompt_ids = [stoi[ch] for ch in prompt if ch in stoi]
    if not prompt_ids:
        prompt_ids = [0]
    seed = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    generated = net.generate(seed, length, temperature=temperature, top_k=top_k)[0].tolist()
    return "".join(itos[i] for i in generated)

print(sample_text("LSTM", prompt="ROMEO:", length=800, temperature=0.8, top_k=20))


In [ ]:
for temp in (0.4, 0.8, 1.2):
    print(f"\n--- temperatura {temp} ---")
    print(sample_text("LSTM", prompt="KING:", length=300, temperature=temp, top_k=30))


## 9. Kwestie praktyczne i warianty

### Zmienne długości i padding

W batchu tekstów różnej długości często dodaje się padding. Zwykłe RNN aktualizuje stan również na paddingu, co może zniekształcić końcowy stan. Użyj `pack_padded_sequence` dla klasyfikacji lub zamaskuj pozycje paddingowe w stracie. Uważaj na `lengths` (CPU dla części API) i sortowanie, jeśli `enforce_sorted=True`.

### Klasyfikacja całej sekwencji

Dla etykiety przypisanej całemu tekstowi pobierz końcowy stan (uwzględniając kierunki i warstwy) albo zastosuj pooling po wyjściach. Przy paddingu trzeba użyć stanu dla prawdziwej długości. Dwukierunkowy GRU/LSTM często pasuje do zadań offline, gdzie cały tekst jest dostępny.

### Przewidywanie wartości ciągłej

Dla prognozowania szeregu czasowego wejściem może być `(B,T,F)` z cechami rzeczywistymi, a głowica liniowa zwraca kolejną wartość. Podział czasowy powinien zachować chronologię. Dopasuj skalowanie wyłącznie na treningu, unikaj przecieku przyszłości, porównuj z naiwnym baseline i oceniaj na horyzoncie interesującym zastosowanie.

### Stateful vs stateless

Tutaj okna są niezależne (stateless). W trenowaniu strumieniowym można zachować stan dla kolejnego fragmentu, ale wtedy batch slot musi kontynuować ten sam strumień. Po zmianie kolejności/mieszaniu przykładów należy wyzerować lub poprawnie przypisać stan. Przenoszenie stanu bez kontroli jest częstym źródłem cichego błędu.

### Wydajność i ograniczenia

Rekurencja uzależnia krok `t` od `t-1`, więc czasowo trudniej ją zrównoleglić niż konwolucje czy Transformer. Embedding pozwala równolegle policzyć reprezentacje wejść, ale same kroki ukryte pozostają sekwencyjne. RNN nadal są przydatne przy strumieniowaniu, małej pamięci, krótkich sekwencjach i istniejących systemach.


## 10. Dobre praktyki i częste błędy

| Problem | Objaw | Sprawdź / popraw |
|---|---|---|
| Złe przesunięcie celu | model kopiuje wejście lub strata nie ma sensu | `y = x` przesunięte o dokładnie jeden token |
| Zła oś tensora | dziwne kształty lub batch traktowany jak czas | ustaw `batch_first=True` albo zmień kolejność osi |
| Softmax przed CrossEntropyLoss | gorsza stabilność i niepoprawna skala | podawaj surowe logits |
| Gradient eksploduje | NaN, nagłe skoki straty | sprawdź dane/LR, clip gradient norm |
| Stan między niepowiązanymi batchami | walidacja niestabilna, przeciek | zeruj stan dla niezależnych okien |
| Bidirectional w generowaniu | przyszły kontekst nieosiągalny podczas inferencji | użyj jednokierunkowego modelu przyczynowego |
| Ocena train zamiast val | zbyt optymistyczne wrażenie | osobny walidacyjny podział chronologiczny |
| Przypadkowe mieszanie chronologii | przeciek w szeregu czasowym | najpierw podział czasowy, potem okna |
| Brak `eval()` / `no_grad()` | dropout aktywny, niepotrzebny graf | przełącz tryb na inferencji i przy ewaluacji |

### Checklist przed treningiem

- Czy `x[:, t]` ma jako cel `x[:, t+1]`?
- Czy podział danych poprzedza budowanie okien?
- Czy kształt logits kończy się wymiarem słownika/klas?
- Czy funkcja straty dostaje oczekiwane osie i typy?
- Czy baseline jest sensowny i wynik walidacyjny jest raportowany?
- Czy seed, tokenizer, split oraz konfiguracja są zapisane?


## Podsumowanie do szybkiej powtórki

- RNN aktualizuje stan: `h_t = f(x_t, h_{t-1})`; wagi są współdzielone w czasie.
- BPTT liczy gradient po rozwiniętym grafie; długi iloczyn Jacobianów prowadzi do zanikania/eksplozji gradientu.
- LSTM ma stan komórki i bramki; GRU ma uproszczony mechanizm bramkowania; wybór zależy od danych i budżetu.
- Dla causal LM wejście i cel różnią się o jeden token; strata liczy się per pozycja.
- `batch_first=True` oznacza `(B,T,F)` dla wejścia i wyjścia, ale stan pozostaje `(layers*directions,B,H)`.
- Clipping ogranicza eksplozję gradientu; walidacja, poprawny split i kontrola stanu chronią przed błędnymi wnioskami.
- RNN-y są sekwencyjne w czasie; bidirectional nie nadaje się do autoregresyjnego generowania.